In [8]:
import pandas as pd

from utils.file_utils import get_tickets_as_df

# Data exploration

In [6]:
tickets_df = get_tickets_as_df()
tickets_df["message_created_on"] = pd.to_datetime(tickets_df["message_created_on"])
tickets_df.head(100)

,ticket_number,ticket_subject,sent_by,ticket_created_on,message_created_on,message,tags
0,1,NaN,Customer,2022-01-18 01:29:01.238861,2022-01-18 01:29:01.238861,"Hello, world!",Testing
1,1,NaN,Customer,2022-01-18 01:29:01.238861,2022-01-18 01:31:08.914884,Is anybody out there?,Testing
2,1,NaN,Agent,2022-01-18 01:29:01.238861,2022-01-18 01:37:46.365888,Ground control to Major Tom,Testing
3,2,NaN,Customer,2022-01-18 04:55:15.792937,2022-01-18 04:55:15.792937,whoa... chat inception,Testing
4,2,NaN,Agent,2022-01-18 04:55:15.792937,2022-01-18 04:55:35.456387,Talking to myself,Testing
...,...,...,...,...,...,...,...
95,31,NaN,Agent,2022-01-31 15:52:37.698712,2022-01-31 15:52:51.080545,df,Testing
96,31,NaN,Customer,2022-01-31 15:52:37.698712,2022-01-31 15:52:53.080650,asd,Testing
97,31,NaN,Customer,2022-01-31 15:52:37.698712,2022-01-31 15:52:53.236093,f,Testing
98,31,NaN,Customer,2022-01-31 15:52:37.698712,2022-01-31 15:52:53.376098,sdf,Testing


In [7]:
tickets_df.dtypes

ticket_number                  int64
ticket_subject                object
sent_by                       object
ticket_created_on             object
message_created_on    datetime64[ns]
message                       object
tags                          object
dtype: object

In [10]:
# Just investigate the tags, need to check if one ticket has multiple tags
tickets_df.groupby("ticket_number")["tags"].unique()
# conclusion:
# one ticket has multiple tags

ticket_number
1                                            [Testing]
2                                            [Testing]
3                                            [Testing]
4                                            [Testing]
5                                            [Testing]
                             ...                      
1525    [Bug,Ticketing - Email, Ticketing - Email,Bug]
1526                                         [Testing]
1527      [Product Question,Ticketing - Slack connect]
1528      [Bug,Customer Profile, Customer Profile,Bug]
1529                                         [Testing]
Name: tags, Length: 1339, dtype: object

In [13]:
# just investigate the tags and the counts
tickets_df["tags"].value_counts()
# conclusion:
# there's a lot of tags with low frequency
# so they are not very useful
# they can be dropped

tags
Testing                                                           1342
Bug                                                                887
Product Question                                                   847
Feature Request                                                    707
Sales                                                              467
                                                                  ... 
Data - Developer API,Integrations - Slack,Bug,Product Question       1
Bug,Product Question,Data - Developer API,Integrations - Slack       1
Product Question,Data - Developer API,Integrations - Slack,Bug       1
Session Recording - Player,Product Question                          1
Product Question,Ticketing - Slack connect                           1
Name: count, Length: 255, dtype: int64

# Feature engineering

In [16]:
cleaned_tickets_df = tickets_df[
    ~tickets_df["tags"].isnull()
]  # remove tags rows with null

cleaned_tickets_df = (
    cleaned_tickets_df.sort_values("message_created_on")
    .groupby("ticket_number")
    .first()
    .reset_index()
)


cleaned_tickets_df = cleaned_tickets_df[
    ~cleaned_tickets_df["message"].isnull()
]  # remove message rows with null

cleaned_tickets_df = cleaned_tickets_df[
    cleaned_tickets_df["tags"].str.contains("Spam")
    | cleaned_tickets_df["tags"].str.contains("Bug")
    | cleaned_tickets_df["tags"].str.contains("Product Question")
    | cleaned_tickets_df["tags"].str.contains("Feature Request")
    | cleaned_tickets_df["tags"].str.contains("Sales")
]  # filter out spam, bug, product question, feature request, sales


cleaned_tickets_df["tags"].value_counts()
# we still can see a lot of tags with low frequency

tags
Product Question                               128
Bug                                            126
Sales                                          117
Feature Request                                101
Spam                                            22
                                              ... 
Feature Request,Compliance - HIPAA               1
Product Question,Chatbot                         1
Feature Request,Ticketing - Delivery status      1
Onboarding,Product Question                      1
Bug,Infra - Search                               1
Name: count, Length: 146, dtype: int64

In [17]:
def map_categories(cat: str) -> str:
    if "Bug" in cat:
        return "Bug"
    elif "Product Question" in cat:
        return "Product Question"
    elif "Feature Request" in cat:
        return "Feature Request"
    elif "Sales" in cat:
        return "Sales"
    elif "Spam" in cat:
        return "Spam"
    else:
        return "Other"


cleaned_tickets_df["tags"] = cleaned_tickets_df["tags"].apply(map_categories)
cleaned_tickets_df["tags"].value_counts()
# we can see that we only have 5 categories
# and they are: Bug, Product Question, Feature Request, Sales, Spam



tags
Feature Request     212
Product Question    201
Bug                 193
Sales               120
Spam                 23
Name: count, dtype: int64

In [18]:
cleaned_tickets_df

,ticket_number,ticket_subject,sent_by,ticket_created_on,message_created_on,message,tags
20,21,None,Customer,2022-01-24 14:19:01.51779,2022-01-24 14:19:01.517790,Hi! What is your pricing? Can I talk with some...,Sales
21,22,None,Customer,2022-01-24 17:52:10.798997,2022-01-24 17:52:10.798997,"Hi, how much is Atlas?",Sales
29,30,None,Customer,2022-01-28 14:53:47.113617,2022-01-28 14:53:47.113617,hey not super urgent but fyi in ramp bank you ...,Feature Request
38,39,None,Customer,2022-02-05 17:00:58.289275,2022-02-05 17:00:58.289275,some minor feedback - i seem to need to sign i...,Feature Request
41,42,None,Customer,2022-02-08 13:17:02.214116,2022-02-08 13:17:02.214116,Hello! Just came across your site: do you offe...,Feature Request
...,...,...,...,...,...,...,...
1329,1522,Chat ticket was not created in Atlas,Customer,2024-05-10 15:03:56.591052,2024-05-10 15:03:56.591052,"Good morning, everyone. A customer contacted u...",Bug
1330,1523,Actions post SLA violations,Customer,2024-05-10 16:15:28.883562,2024-05-10 16:15:28.883562,Hey quick question - we are figuring out how t...,Feature Request
1331,1524,Data is not indexed,Customer,2024-05-10 21:44:14.193959,2024-05-10 21:44:14.193959,Hi - I'm running into an issue where there are...,Bug
1332,1525,Email encoding issue,Customer,2024-05-12 07:37:47.683159,2024-05-12 07:37:47.683159,Hey @Rahul Asati!It looks like we are still ge...,Bug


# Save to csv

In [19]:
cleaned_tickets_df.to_csv("data/cleaned_tickets_v3.csv", index=False)